## Inspect your project files

In [1]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nImportant project paths:")

paths = [
    "ml/route_cost_model.pkl",
    "dataset/processed/flow_analysis.csv",
    "scratch/topology.cc",
    "routing",
    "results"
]

for path in paths:
    print(f"{path}: {'FOUND' if os.path.exists(path) else 'NOT FOUND'}")

Current working directory:
C:\CN\AI-Adaptive-Routing

Important project paths:
ml/route_cost_model.pkl: FOUND
dataset/processed/flow_analysis.csv: FOUND
scratch/topology.cc: NOT FOUND
routing: FOUND
results: FOUND


In [2]:
import joblib

model_path = "ml/route_cost_model.pkl"

if os.path.exists(model_path):
    package = joblib.load(model_path)

    print("Model loaded successfully!")
    print("Model name:", package["model_name"])
    print("Features:", package["features"])
    print("Target:", package["target"])
else:
    print("Model file not found. Check the project folder.")

Model loaded successfully!
Model name: Decision Tree
Features: ['packets_sent', 'packets_received', 'packets_lost', 'delay', 'throughput', 'jitter', 'packet_delivery_ratio_percent']
Target: route_cost


## Step 2: Generate candidate paths through your 10-router topology.

## Step 3: Prepare valid measurements for each candidate route.

## Step 4: Predict route costs and select a candidate.

## Step 5: Connect the decision to NS-3 and test it.

In [3]:
import pandas as pd
import numpy as np
import joblib
import networkx as nx

print("All libraries imported successfully!")

All libraries imported successfully!


## Load the trained model

In [4]:
model_package = joblib.load("ml/route_cost_model.pkl")

model = model_package["model"]
features = model_package["features"]

print("Model:", model_package["model_name"])
print("Required features:", features)

Model: Decision Tree
Required features: ['packets_sent', 'packets_received', 'packets_lost', 'delay', 'throughput', 'jitter', 'packet_delivery_ratio_percent']


## Load the dataset

In [5]:
df = pd.read_csv("dataset/processed/flow_analysis.csv")

print("Dataset rows:", len(df))
print("Dataset columns:", df.columns.tolist())

display(df.head())

Dataset rows: 9
Dataset columns: ['flow_id', 'source', 'destination', 'packets_sent', 'packets_received', 'packets_lost', 'delay', 'throughput', 'jitter', 'traffic_level', 'packet_delivery_ratio_percent', 'route_cost']


,flow_id,source,destination,packets_sent,packets_received,packets_lost,delay,throughput,jitter,traffic_level,packet_delivery_ratio_percent,route_cost
0,1,10.1.2.2,10.1.9.2,9760,9760,0,0.019261,2.585600,4.385920e-05,low,100.0,0.299475
1,2,10.1.1.2,10.1.10.2,5493,5493,0,0.029590,2.714940,0.000000e+00,low,100.0,0.336913
2,3,10.1.9.2,10.1.2.2,4882,4882,0,0.019022,0.119334,3.278020e-10,low,100.0,0.372506
3,1,10.1.2.2,10.1.9.2,38471,38471,0,0.019270,10.191500,4.366710e-05,medium,100.0,0.077082
4,2,10.1.1.2,10.1.10.2,21972,21972,0,0.029590,10.859000,0.000000e+00,medium,100.0,0.118361


## Build the router topology

In [6]:
G = nx.Graph()

router_links = [
    ("R1", "R2"), ("R1", "R3"),
    ("R2", "R4"), ("R3", "R4"),
    ("R2", "R5"), ("R4", "R6"),
    ("R5", "R6"), ("R5", "R7"),
    ("R6", "R8"), ("R7", "R8"),
    ("R7", "R9"), ("R8", "R10"),
    ("R9", "R10")
]

for u, v in router_links:
    G.add_edge(u, v, bandwidth_mbps=100, delay_ms=5)

print("Routers:", G.number_of_nodes())
print("Links:", G.number_of_edges())

Routers: 10
Links: 13


## candidate paths

In [7]:
source = "R1"
destination = "R10"

candidate_paths = list(
    nx.shortest_simple_paths(G, source, destination)
)[:5]

for i, path in enumerate(candidate_paths, start=1):
    print(f"Path {i}: {' -> '.join(path)}")

Path 1: R1 -> R2 -> R4 -> R6 -> R8 -> R10
Path 2: R1 -> R3 -> R4 -> R6 -> R8 -> R10
Path 3: R1 -> R2 -> R5 -> R6 -> R8 -> R10
Path 4: R1 -> R2 -> R5 -> R7 -> R8 -> R10
Path 5: R1 -> R2 -> R5 -> R7 -> R9 -> R10


In [1]:
import pandas as pd
import os

file_path = "dataset/processed/flow_analysis.csv"

df = pd.read_csv(file_path)

print("Total rows:", len(df))
print("\nActual columns:")
print(df.columns.tolist())

print("\nFirst 10 rows:")
display(df.head(10))

print("\nMissing values:")
display(df.isnull().sum())

print("\nData types:")
display(df.dtypes)

Total rows: 9

Actual columns:
['flow_id', 'source', 'destination', 'packets_sent', 'packets_received', 'packets_lost', 'delay', 'throughput', 'jitter', 'traffic_level', 'packet_delivery_ratio_percent', 'route_cost']

First 10 rows:


,flow_id,source,destination,packets_sent,packets_received,packets_lost,delay,throughput,jitter,traffic_level,packet_delivery_ratio_percent,route_cost
0,1,10.1.2.2,10.1.9.2,9760,9760,0,0.019261,2.585600,4.385920e-05,low,100.0,0.299475
1,2,10.1.1.2,10.1.10.2,5493,5493,0,0.029590,2.714940,0.000000e+00,low,100.0,0.336913
2,3,10.1.9.2,10.1.2.2,4882,4882,0,0.019022,0.119334,3.278020e-10,low,100.0,0.372506
3,1,10.1.2.2,10.1.9.2,38471,38471,0,0.019270,10.191500,4.366710e-05,medium,100.0,0.077082
4,2,10.1.1.2,10.1.10.2,21972,21972,0,0.029590,10.859000,0.000000e+00,medium,100.0,0.118361
5,3,10.1.9.2,10.1.2.2,19237,19237,0,0.019022,0.470216,8.317740e-11,medium,100.0,0.361980
6,1,10.1.2.2,10.1.9.2,52194,52194,0,0.019270,13.825800,4.360110e-05,high,100.0,0.077079
7,2,10.1.1.2,10.1.10.2,43945,43945,0,0.029590,21.717900,0.000000e+00,high,100.0,0.118361
8,3,10.1.9.2,10.1.2.2,26099,26099,0,0.019022,0.637946,6.130740e-11,high,100.0,0.356948



Missing values:


flow_id                          0
source                           0
destination                      0
packets_sent                     0
packets_received                 0
packets_lost                     0
delay                            0
throughput                       0
jitter                           0
traffic_level                    0
packet_delivery_ratio_percent    0
route_cost                       0
dtype: int64


Data types:


flow_id                            int64
source                               str
destination                          str
packets_sent                       int64
packets_received                   int64
packets_lost                       int64
delay                            float64
throughput                       float64
jitter                           float64
traffic_level                        str
packet_delivery_ratio_percent    float64
route_cost                       float64
dtype: object

In [2]:
import os
import pandas as pd
import numpy as np
import joblib

data_path = "dataset/processed/flow_analysis.csv"
model_path = "ml/route_cost_model.pkl"

if not os.path.exists(data_path):
    raise FileNotFoundError(f"Dataset not found: {data_path}")

if not os.path.exists(model_path):
    raise FileNotFoundError(f"Trained model not found: {model_path}")

df = pd.read_csv(data_path)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())
display(df.head())

Rows: 9
Columns: ['flow_id', 'source', 'destination', 'packets_sent', 'packets_received', 'packets_lost', 'delay', 'throughput', 'jitter', 'traffic_level', 'packet_delivery_ratio_percent', 'route_cost']


,flow_id,source,destination,packets_sent,packets_received,packets_lost,delay,throughput,jitter,traffic_level,packet_delivery_ratio_percent,route_cost
0,1,10.1.2.2,10.1.9.2,9760,9760,0,0.019261,2.585600,4.385920e-05,low,100.0,0.299475
1,2,10.1.1.2,10.1.10.2,5493,5493,0,0.029590,2.714940,0.000000e+00,low,100.0,0.336913
2,3,10.1.9.2,10.1.2.2,4882,4882,0,0.019022,0.119334,3.278020e-10,low,100.0,0.372506
3,1,10.1.2.2,10.1.9.2,38471,38471,0,0.019270,10.191500,4.366710e-05,medium,100.0,0.077082
4,2,10.1.1.2,10.1.10.2,21972,21972,0,0.029590,10.859000,0.000000e+00,medium,100.0,0.118361


## trained model

In [3]:
model_package = joblib.load(model_path)

model = model_package["model"]
features = model_package["features"]
target = model_package["target"]

print("Model:", model_package["model_name"])
print("Input features:", features)
print("Target:", target)

Model: Decision Tree
Input features: ['packets_sent', 'packets_received', 'packets_lost', 'delay', 'throughput', 'jitter', 'packet_delivery_ratio_percent']
Target: route_cost


In [4]:
missing_features = [col for col in features if col not in df.columns]

if missing_features:
    raise ValueError(
        f"Dataset is missing model features: {missing_features}"
    )

if target not in df.columns:
    raise ValueError(f"Target column '{target}' is missing from dataset.")

print("All required feature columns are present.")
print("Target column is present.")

All required feature columns are present.
Target column is present.


## Prepare actual records for prediction

In [5]:
X = df[features].copy()

for col in features:
    X[col] = pd.to_numeric(X[col], errors="coerce")

valid_rows = X.notna().any(axis=1)
X = X.loc[valid_rows].copy()

# The earlier training workflow used median imputation.
X = X.fillna(X.median())

if X.isna().any().any():
    raise ValueError("Some features contain no usable numeric values.")

print("Records prepared:", len(X))
print("Features used:", X.columns.tolist())
display(X.head())

Records prepared: 9
Features used: ['packets_sent', 'packets_received', 'packets_lost', 'delay', 'throughput', 'jitter', 'packet_delivery_ratio_percent']


,packets_sent,packets_received,packets_lost,delay,throughput,jitter,packet_delivery_ratio_percent
0,9760,9760,0,0.019261,2.585600,4.385920e-05,100.0
1,5493,5493,0,0.029590,2.714940,0.000000e+00,100.0
2,4882,4882,0,0.019022,0.119334,3.278020e-10,100.0
3,38471,38471,0,0.019270,10.191500,4.366710e-05,100.0
4,21972,21972,0,0.029590,10.859000,0.000000e+00,100.0


## run the trained model

In [6]:
predicted_cost = model.predict(X)

prediction_df = df.loc[X.index].copy()
prediction_df["predicted_route_cost"] = predicted_cost

if target in prediction_df.columns:
    prediction_df["prediction_error"] = (
        prediction_df["predicted_route_cost"]
        - pd.to_numeric(prediction_df[target], errors="coerce")
    )

display(prediction_df.head(10))

,flow_id,source,destination,packets_sent,packets_received,packets_lost,delay,throughput,jitter,traffic_level,packet_delivery_ratio_percent,route_cost,predicted_route_cost,prediction_error
0,1,10.1.2.2,10.1.9.2,9760,9760,0,0.019261,2.585600,4.385920e-05,low,100.0,0.299475,0.299475,0.000000
1,2,10.1.1.2,10.1.10.2,5493,5493,0,0.029590,2.714940,0.000000e+00,low,100.0,0.336913,0.299475,-0.037438
2,3,10.1.9.2,10.1.2.2,4882,4882,0,0.019022,0.119334,3.278020e-10,low,100.0,0.372506,0.372506,0.000000
3,1,10.1.2.2,10.1.9.2,38471,38471,0,0.019270,10.191500,4.366710e-05,medium,100.0,0.077082,0.077082,0.000000
4,2,10.1.1.2,10.1.10.2,21972,21972,0,0.029590,10.859000,0.000000e+00,medium,100.0,0.118361,0.118361,0.000000
5,3,10.1.9.2,10.1.2.2,19237,19237,0,0.019022,0.470216,8.317740e-11,medium,100.0,0.361980,0.361980,0.000000
6,1,10.1.2.2,10.1.9.2,52194,52194,0,0.019270,13.825800,4.360110e-05,high,100.0,0.077079,0.077079,0.000000
7,2,10.1.1.2,10.1.10.2,43945,43945,0,0.029590,21.717900,0.000000e+00,high,100.0,0.118361,0.118361,0.000000
8,3,10.1.9.2,10.1.2.2,26099,26099,0,0.019022,0.637946,6.130740e-11,high,100.0,0.356948,0.356948,0.000000


## Save the prediction results

In [7]:
os.makedirs("results/csv", exist_ok=True)

output_path = "results/csv/route_cost_predictions.csv"
prediction_df.to_csv(output_path, index=False)

print("Predictions saved to:", output_path)
print("Total predictions:", len(prediction_df))

Predictions saved to: results/csv/route_cost_predictions.csv
Total predictions: 9


## dataset contains route-specific measurements ?

In [8]:
route_related_columns = [
    col for col in df.columns
    if any(word in col.lower() for word in [
        "route", "path", "source", "destination",
        "router", "link", "delay", "throughput",
        "packets_lost", "jitter"
    ])
]

print("Relevant columns found:")
for col in route_related_columns:
    print("-", col)

print("\nUnique sources:")
if "source" in df.columns:
    print(df["source"].dropna().unique())

print("\nUnique destinations:")
if "destination" in df.columns:
    print(df["destination"].dropna().unique())

Relevant columns found:
- source
- destination
- packets_lost
- delay
- throughput
- jitter
- route_cost

Unique sources:
<StringArray>
['10.1.2.2', '10.1.1.2', '10.1.9.2']
Length: 3, dtype: str

Unique destinations:
<StringArray>
['10.1.9.2', '10.1.10.2', '10.1.2.2']
Length: 3, dtype: str


In [9]:
print("Prediction summary")
print("------------------")
print("Minimum predicted cost:", prediction_df["predicted_route_cost"].min())
print("Maximum predicted cost:", prediction_df["predicted_route_cost"].max())
print("Mean predicted cost:", prediction_df["predicted_route_cost"].mean())
print("Standard deviation:", prediction_df["predicted_route_cost"].std())

Prediction summary
------------------
Minimum predicted cost: 0.077079
Maximum predicted cost: 0.372506
Mean predicted cost: 0.2312518888888889
Standard deviation: 0.12998903846329163


##  Record the current implementation status

In [10]:
status_report = {
    "dataset_loaded": os.path.exists(data_path),
    "trained_model_loaded": os.path.exists(model_path),
    "predictions_generated": len(prediction_df) > 0,
    "prediction_file_saved": os.path.exists(output_path),
    "ns3_adaptive_route_switching_verified": False
}

for task, status in status_report.items():
    print(f"{task}: {'YES' if status else 'NO'}")

dataset_loaded: YES
trained_model_loaded: YES
predictions_generated: YES
prediction_file_saved: YES
ns3_adaptive_route_switching_verified: NO
